# Inside the Data Analytics Agent

A hands-on tutorial for learning how this text-to-SQL agent works from the inside out.

By the end, you will be able to:

1. explain the coordinator/subagent architecture;
2. inspect the source registry, selected OSI model, prompts, memory, and skills;
3. test the backend-neutral SQL safety boundary and result artifact store;
4. construct the real Deep Agent used by the application;
5. observe a LangGraph human-in-the-loop interrupt;
6. approve, edit, or reject generated SQL; and
7. relate the notebook flow to the FastAPI and Streamlit application.

> **Cost and safety:** Building the agent does not call OpenAI. The live exercise is off by default and is clearly marked. SQL is dialect-parsed, human-reviewed, and executed through a source-bound backend with configured timeout and row caps.

## 1. Start with the mental model

```text
User question
     │
     ▼
Data Analytics Agent coordinator
  ├─ conversation-bound source from data_sources.yaml
  ├─ conversation memory: AGENTS.md
  ├─ safe tools: list_conversation_results + inspect_conversation_result
  └─ task → isolated text-to-SQL subagent
                 ├─ selected OSI model (primary context)
                 ├─ query-writing skill
                 ├─ schema-exploration skill
                 ├─ generic metadata/validation fallback tools
                 └─ execute_sql
                         │
                         ▼
                   HITL interrupt
                  approve/edit/reject
                         │
                         ▼
               SQLBackend execution
                 ├─ SQLiteBackend in this POC
                 ├─ ≤10 rows to model
                 └─ ≤500 rows to ResultStore/UI
```

The important design idea is **separation of responsibilities**. The coordinator manages a source-bound conversation. The specialist grounds itself in that source's OSI model and writes SQL. A dependency-injected backend handles metadata, validation, and execution. The human owns the final execution decision. Large data lives outside model context.

## 2. Notebook setup

Run this notebook from the `data-analytics-agent` directory. The setup cell also handles being launched from the repository root. It loads `.env` but never prints your API key.

In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import inspect
import json
import os
import sys
import uuid

import pandas as pd
import yaml
from IPython.display import Markdown, display
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    candidate = PROJECT_ROOT / "data-analytics-agent"
    if (candidate / "pyproject.toml").is_file():
        PROJECT_ROOT = candidate

assert (PROJECT_ROOT / "data_analytics_agent").is_dir(), (
    "Launch Jupyter from data-analytics-agent or its parent repository."
)
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

print(f"Project: {PROJECT_ROOT}")
print(f"OPENAI_API_KEY configured: {bool(os.getenv('OPENAI_API_KEY'))}")

In [ ]:
from data_analytics_agent.backends import create_backend
from data_analytics_agent.config import Settings

settings = Settings()
packages = [
    "deepagents", "langchain", "langgraph", "langchain-openai",
    "sqlglot", "fastapi", "streamlit",
]
versions = {
    package: metadata.version(package)
    for package in packages
    if package in {dist.metadata["Name"] for dist in metadata.distributions()}
}
display(pd.DataFrame(versions.items(), columns=["package", "version"]))
print(f"Model: {settings.model}")
catalog = settings.load_catalog()
selected_source = catalog.get(catalog.default_source_id)
selected_backend = create_backend(selected_source, settings.project_root)
print(f"Registry: {catalog.config_path}")
print(f"Default source: {selected_source.name} ({selected_source.source_id})")
print(f"Backend: {selected_source.backend_type}; dialect: {selected_source.dialect}")
print(f"Semantic model: {selected_source.semantic_model_path}")
print(f"Registered sources: {', '.join(catalog.sources)}")
print(f"Readiness issues: {settings.readiness_errors() or 'none'}")

## 3. Semantic grounding: why the agent reads OSI first

Without a semantic layer, an LLM may repeatedly probe table schemas, guess join paths, or misuse measures at the wrong grain. Every registered source therefore requires an OSI file containing physical sources, friendly concepts, keys, relationships, synonyms, metrics, and AI instructions.

The OSI file is **primary context**. Live schema tools are fallback evidence when the semantic file is ambiguous or appears inconsistent.

In [ ]:
with selected_source.semantic_model_path.open() as handle:
    osi_document = yaml.safe_load(handle)

model = osi_document["semantic_model"][0]
print("OSI version:", osi_document["version"])
print("Semantic model:", model["name"])
print("Datasets:", len(model["datasets"]))
print("Relationships:", len(model["relationships"]))
print("Canonical metrics:", len(model["metrics"]))

In [ ]:
dataset_rows = [
    {
        "semantic_name": dataset["name"],
        "physical_table": dataset["source"],
        "primary_key": ", ".join(dataset["primary_key"]),
        "field_count": len(dataset["fields"]),
    }
    for dataset in model["datasets"]
]
display(pd.DataFrame(dataset_rows))

relationship_rows = [
    {
        "relationship": relation["name"],
        "from": f"{relation['from']}.{','.join(relation['from_columns'])}",
        "to": f"{relation['to']}.{','.join(relation['to_columns'])}",
    }
    for relation in model["relationships"]
]
display(pd.DataFrame(relationship_rows))

In [ ]:
metric_rows = []
for metric in model["metrics"]:
    dialect = metric["expression"]["dialects"][0]
    metric_rows.append({
        "metric": metric["name"],
        "expression": dialect["expression"],
        "meaning": metric["description"],
    })
display(pd.DataFrame(metric_rows))

print("Global AI instructions:\n")
print(model["ai_context"]["instructions"])

## 4. The context stack: prompt, memory, and skills

These three mechanisms solve different problems:

| Mechanism | What it contains | When it is available |
|---|---|---|
| System prompt | Role, delegation rules, output behavior | Always |
| `AGENTS.md` memory | Project-wide operating policy | Loaded into coordinator context |
| Skills | Focused procedures and domain heuristics | Progressively loaded when relevant |

Custom Deep Agent subagents do **not** automatically inherit coordinator skills. This project explicitly assigns both SQL skills to the `text-to-sql` subagent.

In [ ]:
from data_analytics_agent.coordinator import _coordinator_prompt
from data_analytics_agent.agents.text_to_sql.agent import _sql_subagent_prompt

display(Markdown("### Coordinator system prompt"))
print(_coordinator_prompt(selected_source, visualization_enabled=True))
display(Markdown("### Text-to-SQL subagent system prompt"))
print(_sql_subagent_prompt(selected_source))

In [ ]:
print((PROJECT_ROOT / "AGENTS.md").read_text())

In [ ]:
skill_paths = sorted((PROJECT_ROOT / "skills").glob("*/SKILL.md"))
for path in skill_paths:
    display(Markdown(f"### `{path.parent.name}` skill"))
    print(path.read_text())

### What “progressive disclosure” means here

A skill path is registered with the subagent, but the subagent reads the detailed `SKILL.md` only as it works. That keeps the initial prompt smaller while preserving specialized guidance. The SQL prompt also requests a read limit of at least 1,000 lines because Deep Agents' file-reading tool otherwise reads a small default slice.

The project filesystem is mounted virtually at `/project/`. Permissions are allowlisted for `AGENTS.md`, `semantic/**`, and `skills/**`, followed by catch-all deny rules. Permission order matters because the first matching rule wins.

## 5. Structured outputs are contracts

Free-form text is pleasant for a person but unreliable for application code. The specialist and coordinator use strict Pydantic contracts through LangChain's provider-native `ProviderStrategy`. Unknown fields are rejected.

- `QueryResult` is the small executor response visible to the model.
- `SQLAnalysisResult` is the specialist's interpretation.
- `FinalAnswer` is the coordinator's response to the UI.
- `ChartSpec` is the exact reviewed declarative visualization contract.

In [ ]:
from data_analytics_agent.schemas import FinalAnswer, QueryResult, SQLAnalysisResult
from data_analytics_agent.agents.visualization.schemas import ChartSpec

for schema in (QueryResult, SQLAnalysisResult, ChartSpec, FinalAnswer):
    display(Markdown(f"### `{schema.__name__}`"))
    properties = schema.model_json_schema()["properties"]
    display(pd.DataFrame([
        {
            "field": name,
            "type": spec.get("type", spec.get("anyOf", "nested")),
            "description": spec.get("description", ""),
        }
        for name, spec in properties.items()
    ]))

### Visualization contract lab

Visualization is optional and routes only after an explicit chart request. The specialist consumes one saved result, never SQL or arbitrary Python, and proposes exactly one strict `ChartSpec`. Business aggregation stays in reviewed SQL; chart-layer work is limited to presentation sorting/category limits, bar orientation, histogram bins, and box quartiles. `create_chart` has its own approve/edit/reject interrupt before Plotly rendering.

In [ ]:
from data_analytics_agent.agents.visualization.renderer import build_chart
from data_analytics_agent.agents.visualization.validation import validate_chart_spec

chart_spec = ChartSpec(
    result_id=small_result.result_id if 'small_result' in globals() else 'replace-after-section-7',
    chart_type='bar',
    title='Track count by genre',
    x='genre',
    y=['track_count'],
    sort_by='track_count',
    sort_direction='descending',
    category_limit=10,
    orientation='horizontal',
)
display(chart_spec)

## 6. SQL safety is defense in depth

Human review is valuable, but it is not the only safety control:

1. SQLGlot parses exactly one query using the selected dialect.
2. Only SELECT/CTE/set operations pass the shared structural validator.
3. DDL, DML, procedures, metadata, session, and administrative commands are rejected.
4. Human approval is mandatory and edited SQL is validated again.
5. The selected backend adds connection-specific safety controls.
6. SQLite opens with `mode=ro`, denies mutation opcodes, and enforces a progress deadline.
7. Configured executor and model-sample caps bound result size.

The exact reviewed SQL is executed. The executor never silently adds or changes a `LIMIT`.

In [ ]:
from data_analytics_agent.agents.text_to_sql.tools import SQLValidationError, validate_readonly_sql

print(inspect.getsource(validate_readonly_sql))

In [ ]:
examples = {
    "simple SELECT": "SELECT Name FROM Artist ORDER BY Name",
    "CTE": "WITH totals AS (SELECT SUM(Total) AS revenue FROM Invoice) SELECT * FROM totals",
    "set operation": "SELECT Name FROM Artist UNION SELECT Name FROM Genre",
    "write": "UPDATE Artist SET Name = 'unsafe' WHERE ArtistId = 1",
    "multiple statements": "SELECT 1; SELECT 2",
    "PRAGMA": "PRAGMA table_info(Artist)",
}

rows = []
for label, sql in examples.items():
    try:
        expression = validate_readonly_sql(sql, selected_source.dialect)
        rows.append({"example": label, "accepted": True, "result": type(expression).__name__})
    except SQLValidationError as exc:
        rows.append({"example": label, "accepted": False, "result": str(exc)})
display(pd.DataFrame(rows))

## 7. The deterministic executor and result artifact

The next cell calls the guarded executor directly so you can study its data contract without paying for a model call. In the real agent, this same operation is wrapped as the `execute_sql` tool and is protected by HITL middleware.

Direct invocation here bypasses only the approval pause—not the parser, read-only connection, authorizer, timeout, or row cap.

In [ ]:
from data_analytics_agent.agents.text_to_sql.tools import execute_query
from data_analytics_agent.stores import ResultStore

lab_store = ResultStore()
lab_thread_id = "notebook-safety-lab"
lab_sql = """
SELECT g.Name AS genre, COUNT(t.TrackId) AS track_count
FROM Genre AS g
LEFT JOIN Track AS t ON t.GenreId = g.GenreId
GROUP BY g.GenreId, g.Name
ORDER BY track_count DESC, genre ASC
""".strip()

small_result = execute_query(
    backend=selected_backend,
    source=selected_source,
    query=lab_sql,
    thread_id=lab_thread_id,
    result_store=lab_store,
)
display(pd.DataFrame(small_result.sample_rows))
display(small_result)

In [ ]:
artifact = lab_store.get(small_result.result_id, lab_thread_id, source_id=selected_source.source_id)
page = lab_store.page(small_result.result_id, lab_thread_id, source_id=selected_source.source_id, offset=10, limit=10)

print(f"Rows visible to the model: {len(small_result.sample_rows)}")
print(f"Rows saved for the application: {artifact.row_count}")
print(f"Second page rows: {len(page.rows)}")
display(pd.DataFrame(page.rows))

# Thread scoping prevents another conversation from reading the artifact.
try:
    lab_store.get(small_result.result_id, "another-thread")
except KeyError:
    print("Thread isolation check passed.")

## 8. Build the real Deep Agent

This creates the same compiled graph used by FastAPI. It does **not** send a request to OpenAI.

Important constructor choices:

- the default general-purpose subagent is disabled;
- the custom `text-to-sql` subagent is always registered;
- the `data-visualization` subagent is registered only when its feature flag is enabled;
- each custom subagent gets its own tools, permissions, prompt, and output schema;
- `execute_sql` and `create_chart` have `approve`, `edit`, and `reject` interrupt decisions;
- `InMemorySaver` checkpoints paused state by `thread_id`; and
- the coordinator exposes only bounded result discovery and `head(10)` inspection tools directly.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from data_analytics_agent.coordinator import build_agent

agent_result_store = ResultStore()
checkpointer = InMemorySaver()
agent = build_agent(
    settings,
    agent_result_store,
    source=selected_source,
    backend=selected_backend,
    checkpointer=checkpointer,
)

print("Agent name:", agent.name)
print("Compiled graph nodes:", sorted(agent.nodes))

In [ ]:
# This is the actual application constructor, shown as executable documentation.
print(inspect.getsource(build_agent))

## 9. Run the full agent and pause before SQL

Set `RUN_LIVE_AGENT = True` when you are ready. The first model run should stop at an `execute_sql` interrupt. It must not execute the database query yet.

The checkpoint key is the run ID supplied as LangGraph's configurable `thread_id`. A resume must use the **same** configuration. Typed graph state carries the application conversation, run, and source IDs into inline subagent tools.

In [ ]:
RUN_LIVE_AGENT = False  # Change to True to call OpenAI.

question = "Which five artists generated the most line-item revenue?"
live_thread_id = f"notebook-{uuid.uuid4()}"
live_run_id = f"run-{uuid.uuid4()}"
live_config = {"configurable": {"thread_id": live_run_id}}

print("Question:", question)
print("Checkpoint thread:", live_thread_id)
print("Live call enabled:", RUN_LIVE_AGENT)

In [ ]:
from data_analytics_agent.run_manager import _extract_approval

first_output = None
approval = None

if RUN_LIVE_AGENT:
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Add OPENAI_API_KEY to .env before enabling the live lab.")
    first_output = agent.invoke(
        {
            "messages": [{"role": "user", "content": question}],
            "thread_id": live_thread_id,
            "run_id": live_run_id,
            "source_id": selected_source.source_id,
        },
        config=live_config,
        version="v2",
    )
    interrupts = list(getattr(first_output, "interrupts", []))
    approval = _extract_approval(interrupts, source=selected_source)
    print("The graph paused before execution.")
    print("Allowed decisions:", approval.allowed_decisions)
    print("\nGenerated SQL:\n")
    print(approval.query)
else:
    print("Skipped. Set RUN_LIVE_AGENT=True, then rerun this cell.")

In [ ]:
if first_output is not None:
    for m in first_output.value["messages"]:
        m.pretty_print()
else:
    print("Skipped until the live lab is enabled.")

### Review it like an analyst

Before approving, ask:

- Does the SQL answer the actual question?
- Are joins consistent with the OSI relationships?
- Is revenue defined at the correct grain?
- Could a join duplicate invoice totals?
- Is the result ordered and limited as requested?

The decision translator validates both approved and edited SQL, then creates LangGraph's required shape: `Command(resume={"decisions": [...]})`. For an edit, the new call is supplied under `edited_action`; the reviewed action order is preserved.

In [ ]:
from data_analytics_agent.run_manager import decisions_to_command
from data_analytics_agent.schemas import Decision

# Choose one: "approve", "edit", or "reject".
REVIEW_ACTION = "approve"
EDITED_SQL = approval.query if approval else "SELECT Name FROM Artist ORDER BY Name"
REJECTION_FEEDBACK = "Revise the query to use line-level revenue and return five artists."

resume_command = None
if approval:
    if REVIEW_ACTION == "approve":
        decision = Decision(action="approve")
    elif REVIEW_ACTION == "edit":
        decision = Decision(action="edit", edited_sql=EDITED_SQL)
    elif REVIEW_ACTION == "reject":
        decision = Decision(action="reject", feedback=REJECTION_FEEDBACK)
    else:
        raise ValueError("REVIEW_ACTION must be approve, edit, or reject")
    resume_command = decisions_to_command(approval, [decision])
    print(resume_command)
else:
    print("No pending approval. Run the live cell first.")

In [ ]:
completed_output = None
final_answer = None

if RUN_LIVE_AGENT and resume_command is not None:
    completed_output = agent.invoke(
        resume_command,
        config=live_config,
        version="v2",
    )
    new_interrupts = list(getattr(completed_output, "interrupts", []))
    if new_interrupts:
        next_approval = _extract_approval(new_interrupts, source=selected_source)
        print("The agent requested another review cycle.")
        print(next_approval.query)
    else:
        state = completed_output.value if hasattr(completed_output, "value") else completed_output
        final_answer = state.get("structured_response")
        display(final_answer)
else:
    print("Skipped until the live lab is enabled and an approval exists.")

In [ ]:
if final_answer and final_answer.result_id:
    saved = agent_result_store.get(final_answer.result_id, live_thread_id, source_id=selected_source.source_id)
    print("Exact executed SQL:\n")
    print(saved.executed_sql)
    print(f"\nStored rows: {saved.row_count}; truncated: {saved.truncated}")
    display(pd.DataFrame(saved.rows, columns=saved.columns))
else:
    print("A completed live SQL run will display its full stored artifact here.")

### What happens on rejection?

A rejection does not execute SQL. Feedback is returned to the subagent, which can revise its plan and submit another `execute_sql` action. That produces a **new interrupt**. Repeat the review cells with the new approval. This is why the UI supports repeated approval cycles rather than assuming one review per message.

### What happens on edit?

The edited SQL is validated before resume. If valid, the middleware replaces the pending tool arguments. The exact edited text is the SQL executed and later shown in the UI.

## 10. Observability without exposing hidden reasoning

The production run manager consumes `astream_events(version="v3")` internally. It converts raw tool lifecycle events into safe labels such as “Inspecting the OSI semantic model” and “Checking generated SQL.” It deliberately does **not** send prompts, hidden reasoning, tool payloads, or raw database rows to the browser.

In [ ]:
from data_analytics_agent.run_manager import _activity_for_tool

sample_tool_events = [
    ("task", {}),
    ("read_file", {"file_path": selected_source.semantic_virtual_path}),
    ("read_file", {"file_path": "/project/skills/query-writing/SKILL.md"}),
    ("write_todos", {}),
    ("get_table_schema", {}),
    ("validate_sql", {}),
    ("execute_sql", {"query": "SELECT secret-looking-payload"}),
]
display(pd.DataFrame([
    {"tool": name, "sanitized_activity": _activity_for_tool(name, payload)}
    for name, payload in sample_tool_events
]))

## 11. How the notebook maps to FastAPI and Streamlit

The browser does not call the graph directly. FastAPI owns the validated source catalog plus process-local conversations, runs, checkpoints, and results. Streamlit selects a ready source, starts a source-bound conversation, and polls runs without streaming model tokens.

```text
queued → running → approval_required → running → completed
                    │                     │
                    └── reject/replan ────┘
Any active state can become failed.
```

At `approval_required`, Streamlit renders pending SQL and source-specific dialect/limits in an editable text area. The user's decision is posted to FastAPI, translated into a LangGraph `Command`, and resumed against the same checkpoint thread. Changing the source starts a new conversation.

In [ ]:
from data_analytics_agent.api import Services, create_app

tutorial_app = create_app(Services(settings=settings))
route_rows = []
for route in tutorial_app.routes:
    if getattr(route, "path", "").startswith(("/api", "/health")):
        route_rows.append({
            "methods": ", ".join(sorted(route.methods or [])),
            "path": route.path,
            "name": route.name,
        })
display(pd.DataFrame(route_rows))

## 12. Core design lessons

1. **Context engineering beats blind exploration.** Put stable business meaning in an OSI semantic model and reserve live schema tools for fallback.
2. **Isolate specialists.** Give each custom subagent only the prompt, skills, tools, permissions, and response contract it needs.
3. **Use progressive disclosure.** Keep broad prompts concise; load procedural skills when relevant.
4. **Checkpoint before human decisions.** HITL only works reliably when pause and resume share the same checkpointer and thread ID.
5. **Treat structured output as an interface.** Models produce typed data that application code can validate and render.
6. **Keep data artifacts outside context.** Give the model a result ID and a small sample; give the UI the complete capped artifact.
7. **Use deterministic safety controls.** Prompts and human review complement dialect parsing and adapter-specific read-only controls, timeouts, and caps; they do not replace them.
8. **Sanitize observability.** Show useful progress without leaking prompts, hidden reasoning, SQL payloads, or result rows.
9. **Make assumptions explicit.** The answer should state material assumptions and a concise interpretation alongside the data.
10. **Test lifecycle edges.** Source binding, approval, edits, rejection loops, same-thread resume, truncation, result provenance, refresh, and concurrent-run rejection matter as much as the happy path.

## 13. Suggested exercises

Try these in order:

1. Change the direct executor query and observe the 10-row model sample versus the full artifact.
2. Add a harmless semicolon and then a second statement to see the validator's boundary.
3. Enable the live lab and approve the generated SQL unchanged.
4. Start a fresh live thread, edit the SQL before execution, and confirm `executed_sql` exactly matches your edit.
5. Start another fresh thread, reject with specific feedback, and inspect the next proposed query.
6. Ask a follow-up about a saved `result_id` and observe bounded conversation-result discovery instead of a fresh query where appropriate.
7. Switch from Chinook to Financial and observe that Streamlit starts a new source-bound conversation.
8. Modify a synonym in the selected OSI file and test how the agent interprets that phrase.
9. Write a complex analytical question and watch for `write_todos` in the Streamlit activity panel.

For production, the main missing pieces are durable storage/checkpoints, authentication and tenant isolation, stronger resource governance, deployment hardening, evaluation datasets, and operational tracing.

## 14. Verify the implementation

The final cell runs the automated suite. The live OpenAI smoke test remains skipped unless its explicit test conditions are enabled.

In [ ]:
import subprocess

completed = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
completed.check_returncode()

## 15. Continue learning

- [Deep Agents overview](https://docs.langchain.com/oss/python/deepagents/overview)
- [Deep Agents subagents](https://docs.langchain.com/oss/python/deepagents/subagents)
- [Deep Agents event streaming](https://docs.langchain.com/oss/python/deepagents/event-streaming)
- [LangChain human-in-the-loop middleware](https://docs.langchain.com/oss/python/langchain/human-in-the-loop)
- [LangChain structured output](https://docs.langchain.com/oss/python/langchain/structured-output)
- [Datawhale Deep Agents in Action: task planning](https://datawhalechina.github.io/deepagents-in-action/chapters/ch04-task-planning/)
- [Datawhale Deep Agents in Action: skills](https://datawhalechina.github.io/deepagents-in-action/chapters/ch07-skills/)
- [Datawhale Deep Agents in Action: human in the loop](https://datawhalechina.github.io/deepagents-in-action/chapters/ch09-human-in-the-loop/)
- [Apache Ossie/OSI semantic-model specification](https://github.com/apache/ossie/blob/main/core-spec/spec.md)
- [OpenAI GPT-5.4 nano](https://developers.openai.com/api/docs/models/gpt-5.4-nano)

The local `.codex/skills/langchain-dev-guide` also records version-sensitive Deep Agents and LangGraph engineering pitfalls used while building this tutorial.